In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth sacrebleu
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import os
import gc
import torch
import sacrebleu
import time
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

In [ ]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

dataset = load_dataset("HizkiaJ/qed_sermon_100k", split="test", token=hf_token)

texts_id = dataset['text_id']
references = dataset['text_en']
refs_formatted = [references]

df_results = pd.DataFrame({
    'Source (ID)': texts_id,
    'Reference (EN)': references
})

In [ ]:
evaluation_metrics = [] 

models_config = [
    {
        "name": "LLaMA-3.1-8B-Instruct",
        "lora_path": "/kaggle/input/datasets/hizkiajustine/epoch-1-llama-3/checkpoint-2298/checkpoint-2298", 
        "rslora_path": "/kaggle/input/datasets/hizkiajustine/llama-3-1-instruct-rslora-weight/checkpoint-2298", 
        "is_thinking": False
    },
    {
        "name": "Qwen3-8B",
        "lora_path": "/kaggle/input/datasets/hizkiajh/qwen-3-epoch-1/checkpoint-2298/checkpoint-2298", 
        "rslora_path": "/kaggle/input/datasets/hizkiajustine/qwen3-rslora-weight/checkpoint-2298", 
        "is_thinking": True
    },
    {
        "name": "Qwen2.5-7B-Instruct",
        "lora_path": "/kaggle/input/datasets/hizkiajustine/qwen2-5-7b-instruct-lora-weight/checkpoint-2298/checkpoint-2298", 
        "rslora_path": "/kaggle/input/datasets/hizkiajh/qwen-2-5-7b-instruct-rslora-weight/checkpoint-2298/checkpoint-2298", 
        "is_thinking": False
    }
]

def generate_translations(model_instance, tokenizer_instance, texts, is_thinking, batch_size=8):
    predictions = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch_texts = texts[i : i + batch_size]
        
        messages_batch = [
            [
                {"role": "system", "content": "You are a professional translator. Translate the following Indonesian text to English. Provide only the translation, without any explanations or additional text."},
                {"role": "user", "content": text}
            ] for text in batch_texts
        ]
        
        if is_thinking:
            prompts = tokenizer_instance.apply_chat_template(
                messages_batch, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        else:
            prompts = tokenizer_instance.apply_chat_template(
                messages_batch, tokenize=False, add_generation_prompt=True
            )
        
        inputs = tokenizer_instance(prompts, return_tensors="pt", padding=True).to("cuda:0")
        
        with torch.no_grad():
            outputs = model_instance.generate(
                **inputs, 
                max_new_tokens=512,
                use_cache=True,
                do_sample=False,
                pad_token_id=tokenizer_instance.pad_token_id
            )
            
        prompt_length = inputs["input_ids"].shape[1]
        generated_tokens = outputs[:, prompt_length:]
        
        decoded = tokenizer_instance.batch_decode(generated_tokens, skip_special_tokens=False)
        
        for text in decoded:
            if is_thinking and "</think>" in text:
                text = text.split("</think>")[-1]
            
            if tokenizer_instance.pad_token:
                text = text.replace(tokenizer_instance.pad_token, "")
                
            text = (text.replace("<|PAD_TOKEN|>", "")
                        .replace("<|endoftext|>", "")
                        .replace(tokenizer_instance.eos_token, "")
                        .replace("<|im_end|>", "")
                        .replace("<|end_of_sentence|>", "")
                        .strip())
            
            predictions.append(text)
        
    return predictions

for config in models_config:
    model_name = config["name"]
    lora_path = config["lora_path"]
    rslora_path = config["rslora_path"]
    is_thinking = config["is_thinking"]
    
    print(f"\n{'='*50}")
    print(f"MEMULAI EVALUASI MODEL: {model_name}")
    print(f"{'='*50}")
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = lora_path,
        max_seq_length = 512,
        dtype = None,
        load_in_4bit = True,
        token = hf_token,
        device_map = "cuda:0"
    )
    tokenizer.padding_side = "left" 
    FastLanguageModel.for_inference(model)
    
    # Evaluasi Base Model
    print(f"[{model_name}] Menjalankan Inferensi Base Model")
    start_time_base = time.time()
    with model.disable_adapter():
        base_preds = generate_translations(model, tokenizer, texts_id, is_thinking, batch_size=8)
    base_duration = time.time() - start_time_base
    base_bleu = sacrebleu.corpus_bleu(base_preds, refs_formatted).score
    
    # Evaluasi LoRA Model
    print(f"[{model_name}] Menjalankan Inferensi LoRA Model")
    start_time_lora = time.time()
    lora_preds = generate_translations(model, tokenizer, texts_id, is_thinking, batch_size=8)
    lora_duration = time.time() - start_time_lora
    lora_bleu = sacrebleu.corpus_bleu(lora_preds, refs_formatted).score
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    print(f"[{model_name}] Menjalankan Inferensi rsLoRA Model")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = rslora_path,
        max_seq_length = 512,
        dtype = None,
        load_in_4bit = True,
        token = hf_token,
        device_map = "cuda:0"
    )
    tokenizer.padding_side = "left" 
    FastLanguageModel.for_inference(model)

    # Evaluasi rsLoRA Model
    start_time_rslora = time.time()
    rslora_preds = generate_translations(model, tokenizer, texts_id, is_thinking, batch_size=8)
    rslora_duration = time.time() - start_time_rslora
    rslora_bleu = sacrebleu.corpus_bleu(rslora_preds, refs_formatted).score
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    evaluation_metrics.append({
        "Model": model_name,
        "Base BLEU": round(base_bleu, 2),
        "LoRA BLEU": round(lora_bleu, 2),
        "rsLoRA BLEU": round(rslora_bleu, 2),
        "Base Time (detik)": round(base_duration, 2),
        "LoRA Time (detik)": round(lora_duration, 2),
        "rsLoRA Time (detik)": round(rslora_duration, 2)
    })
    
    df_results[f'{model_name} (Base)'] = base_preds
    df_results[f'{model_name} (LoRA)'] = lora_preds
    df_results[f'{model_name} (rsLoRA)'] = rslora_preds

print("="*70)
print("REKAPITULASI METRIK KESELURUHAN (BLEU & WAKTU INFERENSI)")
print("="*70)
df_scores = pd.DataFrame(evaluation_metrics)
print(df_scores.to_string(index=False))

df_scores.to_csv("rekap_metrik_evaluasi.csv", index=False)
df_results.to_csv("komparasi_teks_lengkap.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df_scores = pd.read_csv("rekap_metrik_evaluasi.csv")

sns.set_theme(style="whitegrid")

# GRAFIK 1: PERBANDINGAN BASE VS LORA
fig1, axes1 = plt.subplots(1, 2, figsize=(16, 6))

df_bleu_lora = df_scores[['Model', 'Base BLEU', 'LoRA BLEU']].melt(
    id_vars='Model', var_name='Tipe Model', value_name='Skor BLEU'
)
df_time_lora = df_scores[['Model', 'Base Time (detik)', 'LoRA Time (detik)']].melt(
    id_vars='Model', var_name='Tipe Model', value_name='Waktu (Detik)'
)

# Plot BLEU (Base vs LoRA)
sns.barplot(data=df_bleu_lora, x='Model', y='Skor BLEU', hue='Tipe Model', ax=axes1[0], palette='mako')
axes1[0].set_title('Komparasi Performa BLEU\n(Base vs LoRA)', fontsize=14, fontweight='bold')
axes1[0].set_ylabel('Skor BLEU', fontsize=12)
axes1[0].set_xlabel('Model', fontsize=12)
axes1[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, title=None, fontsize=11)
for container in axes1[0].containers:
    axes1[0].bar_label(container, fmt='%.2f', padding=3, fontsize=11)

# Plot Waktu (Base vs LoRA)
sns.barplot(data=df_time_lora, x='Model', y='Waktu (Detik)', hue='Tipe Model', ax=axes1[1], palette='flare')
axes1[1].set_title('Komparasi Kecepatan Inferensi\n(Base vs LoRA)', fontsize=14, fontweight='bold')
axes1[1].set_ylabel('Total Waktu (Detik)', fontsize=12)
axes1[1].set_xlabel('Model', fontsize=12)
axes1[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, title=None, fontsize=11)
for container in axes1[1].containers:
    axes1[1].bar_label(container, fmt='%.1f', padding=3, fontsize=11)

plt.tight_layout()
fig1.savefig('visualisasi_base_vs_lora.png', dpi=300, bbox_inches='tight')
plt.show()

# GRAFIK 2: PERBANDINGAN BASE VS rsLORA
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))

df_bleu_rslora = df_scores[['Model', 'Base BLEU', 'rsLoRA BLEU']].melt(
    id_vars='Model', var_name='Tipe Model', value_name='Skor BLEU'
)
df_time_rslora = df_scores[['Model', 'Base Time (detik)', 'rsLoRA Time (detik)']].melt(
    id_vars='Model', var_name='Tipe Model', value_name='Waktu (Detik)'
)

# Plot BLEU (Base vs rsLoRA)
sns.barplot(data=df_bleu_rslora, x='Model', y='Skor BLEU', hue='Tipe Model', ax=axes2[0], palette='mako')
axes2[0].set_title('Komparasi Performa BLEU\n(Base vs rsLoRA)', fontsize=14, fontweight='bold')
axes2[0].set_ylabel('Skor BLEU', fontsize=12)
axes2[0].set_xlabel('Model', fontsize=12)
axes2[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, title=None, fontsize=11)
for container in axes2[0].containers:
    axes2[0].bar_label(container, fmt='%.2f', padding=3, fontsize=11)

# Plot Waktu (Base vs rsLoRA)
sns.barplot(data=df_time_rslora, x='Model', y='Waktu (Detik)', hue='Tipe Model', ax=axes2[1], palette='flare')
axes2[1].set_title('Komparasi Kecepatan Inferensi\n(Base vs rsLoRA)', fontsize=14, fontweight='bold')
axes2[1].set_ylabel('Total Waktu (Detik)', fontsize=12)
axes2[1].set_xlabel('Model', fontsize=12)
axes2[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, title=None, fontsize=11)
for container in axes2[1].containers:
    axes2[1].bar_label(container, fmt='%.1f', padding=3, fontsize=11)

plt.tight_layout()
fig2.savefig('visualisasi_base_vs_rslora.png', dpi=300, bbox_inches='tight')
plt.show()